<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/02_risk_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model A — Visit Risk Classification
### Business Purpose: Predict whether a hospital visit represents a Low, Medium, or High operational and clinical risk.

   * Define the target variable as risk_score.
   *  Select and justify feature set based on business relevance.
   *  Perform a time-based train and test split (earliest 80 percent for training, latest 20 percent for testing).
   *  Train a baseline model using Logistic Regression.
   *  Train an advanced model such as Random Forest or Gradient Boosting.
   *  Perform optional hyperparameter tuning and document results.

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [38]:
# Upload file to google.colab
import io
from google.colab import files, drive

drive.mount('/content/drive')

# comment this code, if files already uploaded.
# uploaded = files.upload()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Load data into Python for analysis. Combine them


In [39]:
# Load and read patients, visits & billing csv
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
# patients_df.head()

visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
# visits_df.head()

billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
# billing_df.head()

In [40]:
# Merge dataframe
df_merged = pd.merge(patients_df, visits_df, on='patient_id', how = 'inner')
df_merged = pd.merge(df_merged, billing_df, on='visit_id', how = 'inner')
df_merged.head()


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14,4327,2025-10-15,General,ICU,6.39,High,162,4327,26322.05,13938.52,Pending,NaN,2025-11-29
1,1,53,M,Hyderabad,SecureLife,0,2025-05-14,5395,2025-09-21,Orthopedics,OPD,39.12,Low,181,5395,13258.56,0.00,Rejected,3.0,2025-08-15
2,1,53,M,Hyderabad,SecureLife,0,2025-05-14,9032,2025-02-09,Cardiology,OPD,19.57,Medium,121,9032,26555.09,26555.09,Paid,4.0,2025-12-31
3,1,53,M,Hyderabad,SecureLife,0,2025-05-14,9506,2025-10-09,Neurology,ER,30.32,Medium,135,9506,11583.16,8065.34,Pending,6.0,2025-01-30
4,1,53,M,Hyderabad,SecureLife,0,2025-05-14,10868,2025-07-07,ICU,ER,30.07,High,164,10868,9049.72,9049.72,Paid,19.0,2025-02-26


In [41]:
# Define the target variable as risk_score.
df_target = df_merged[['risk_score']]
display(df_target.head().reset_index())

# Select and justify feature set based on business relevance.
# 'patient_id' not dropped as it will be used for grouping and sorting later
df_features = df_merged.drop(columns=['risk_score', 'visit_id','bill_id', 'doctor_id'])
display(df_features.head())

,index,risk_score
0,0,High
1,1,Low
2,2,Medium
3,3,Medium
4,4,High


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_date,department,visit_type,length_of_stay_hours,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-15,General,ICU,6.39,26322.05,13938.52,Pending,NaN,2025-11-29
1,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-09-21,Orthopedics,OPD,39.12,13258.56,0.00,Rejected,3.0,2025-08-15
2,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-02-09,Cardiology,OPD,19.57,26555.09,26555.09,Paid,4.0,2025-12-31
3,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-09,Neurology,ER,30.32,11583.16,8065.34,Pending,6.0,2025-01-30
4,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-07-07,ICU,ER,30.07,9049.72,9049.72,Paid,19.0,2025-02-26


In [42]:
# Feature Engineering

# Converting date columns to date type
date_columns = ['registration_date', 'visit_date', 'billing_date']
for col in date_columns:
    df_features[col] = pd.to_datetime(df_features[col])

# Handling null/NAN to relevant value
null_counts = df_features.isnull().sum()
columns_with_nulls = null_counts[null_counts > 0]

if not columns_with_nulls.empty:
    print("Columns with null/NaN values and their counts:")
    display(columns_with_nulls)
else:
    print("No columns with null/NaN values found.")


display(df_features.head())

Columns with null/NaN values and their counts:


,0
approved_amount,1318
payment_days,790


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_date,department,visit_type,length_of_stay_hours,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-15,General,ICU,6.39,26322.05,13938.52,Pending,NaN,2025-11-29
1,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-09-21,Orthopedics,OPD,39.12,13258.56,0.00,Rejected,3.0,2025-08-15
2,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-02-09,Cardiology,OPD,19.57,26555.09,26555.09,Paid,4.0,2025-12-31
3,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-09,Neurology,ER,30.32,11583.16,8065.34,Pending,6.0,2025-01-30
4,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-07-07,ICU,ER,30.07,9049.72,9049.72,Paid,19.0,2025-02-26


In [43]:
# Impute 'approved_amount' and 'payment_days' with their medians and create missing indicators

# Create missing indicator for 'approved_amount'
df_features['approved_amount_missing'] = df_features['approved_amount'].isnull().astype(int)
# Calculate median for 'approved_amount' excluding NaNs
median_approved_amount = df_features['approved_amount'].median()
df_features['approved_amount'].fillna(median_approved_amount, inplace=True)

# Create missing indicator for 'payment_days'
df_features['payment_days_missing'] = df_features['payment_days'].isnull().astype(int)
# Calculate median for 'payment_days' excluding NaNs
median_payment_days = df_features['payment_days'].median()
df_features['payment_days'].fillna(median_payment_days, inplace=True)

print("Null values after imputation:")
display(df_features.isnull().sum()[df_features.isnull().sum() > 0])

display(df_features.head())

Null values after imputation:


/tmp/ipython-input-323/3020004162.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_features['approved_amount'].fillna(median_approved_amount, inplace=True)
/tmp/ipython-input-323/3020004162.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].

,0


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_date,department,visit_type,length_of_stay_hours,billed_amount,approved_amount,claim_status,payment_days,billing_date,approved_amount_missing,payment_days_missing
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-15,General,ICU,6.39,26322.05,13938.52,Pending,13.0,2025-11-29,0,1
1,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-09-21,Orthopedics,OPD,39.12,13258.56,0.00,Rejected,3.0,2025-08-15,0,0
2,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-02-09,Cardiology,OPD,19.57,26555.09,26555.09,Paid,4.0,2025-12-31,0,0
3,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-10-09,Neurology,ER,30.32,11583.16,8065.34,Pending,6.0,2025-01-30,0,0
4,1,53,M,Hyderabad,SecureLife,0,2025-05-14,2025-07-07,ICU,ER,30.07,9049.72,9049.72,Paid,19.0,2025-02-26,0,0


In [44]:
#Perform a time-based train and test split (earliest 80 percent for training, latest 20 percent for testing).
#Since this is temporal data, data needs to be sorted for correct train_test_split and modelling.
# Following Dates are selected for sorting -> registration_date, visit_date

df_features_sorted = df_features.sort_values(by=['patient_id','registration_date','visit_date'])
train_X,test_x, train_y, test_y = train_test_split(df_features_sorted, df_target, test_size=0.2, shuffle=False)

In [ ]:
# Preprocessing for Logistic Regression


# Identify categorical columns for one-hot encoding
categorical_cols = train_X.select_dtypes(include='object').columns

# Drop date columns as they cannot be directly used by Logistic Regression without further feature engineering
# The time-based split already implicitly uses the temporal information for splitting.
date_cols = train_X.select_dtypes(include=['datetime64[ns]']).columns

# Create copies for preprocessing
train_X_preprocessed = train_X.drop(columns=date_cols).copy()
test_x_preprocessed = test_x.drop(columns=date_cols).copy()

# Apply one-hot encoding
train_X_preprocessed = pd.get_dummies(train_X_preprocessed, columns=categorical_cols, drop_first=True)
test_x_preprocessed = pd.get_dummies(test_x_preprocessed, columns=categorical_cols, drop_first=True)

# Align columns after one-hot encoding to ensure both train and test sets have the same features
# This handles cases where a category might exist in one set but not the other
missing_in_test = set(train_X_preprocessed.columns) - set(test_x_preprocessed.columns)
for c in missing_in_test:
    test_x_preprocessed[c] = 0

missing_in_train = set(test_x_preprocessed.columns) - set(train_X_preprocessed.columns)
for c in missing_in_train:
    train_X_preprocessed[c] = 0

# Ensure the order of columns is the same
test_x_preprocessed = test_x_preprocessed[train_X_preprocessed.columns]

# Encode the target variable
le = LabelEncoder()
train_y_encoded = le.fit_transform(train_y.values.ravel())
test_y_encoded = le.transform(test_y.values.ravel())

# Update train_X, test_x, train_y, test_y with processed data
train_X = train_X_preprocessed
test_x = test_x_preprocessed
train_y = train_y_encoded
test_y = test_y_encoded

print("Shape of preprocessed train_X:", train_X.shape)
print("Shape of preprocessed test_x:", test_x.shape)
print("Unique values in encoded train_y:", pd.Series(train_y).unique())